# Employees: Bronze -> Silver

Drop bad rows, normalize types and text.

In [1]:
%run ../00_config.ipynb
%run ../00_utils.ipynb

/usr/local/lib/python3.12/site-packages/nbformat/validator.py:434: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  _validate(nbdict, ref, version, version_minor, relax_add_props)


[08/23/26 15:05:55] INFO     Using                                                                  ]8;id=16264419;file:///usr/local/lib/python3.12/site-packages/kedro/framework/project/__init__.py\__init__.py]8;;\:]8;id=16264420;file:///usr/local/lib/python3.12/site-packages/kedro/framework/project/__init__.py#302\302]8;;\
                             '/usr/local/lib/python3.12/site-packages/kedro/framework/project/rich_                
                             logging.yml' as logging configuration.                                                

[08/23/26 15:05:55] WARNING  /usr/local/lib/python3.12/site-packages/kedro/framework/context/contex ]8;id=16264427;file:///usr/local/lib/python3.12/warnings.py\warnings.py]8;;\:]8;id=16264428;file:///usr/local/lib/python3.12/warnings.py#112\112]8;;\
                             t.py:221: UserWarning: Parameters not found in your Kedro project                     
                             config.                                                                               
                             No files of YAML or JSON format found in /app/conf/base or                            
                             /app/conf/local matching the glob pattern(s): ['parameters*',                         
                             'parameters*/**', '**/parameters*']                                                   
                               warn(f"Parameters not found in your Kedro project config.\n{exc!s}")                
                                                                                                                   

[08/23/26 15:05:56] INFO     No typed parameter requirements found, returning original   ]8;id=16264435;file:///usr/local/lib/python3.12/site-packages/kedro/validation/parameter_validator.py\parameter_validator.py]8;;\:]8;id=16264436;file:///usr/local/lib/python3.12/site-packages/kedro/validation/parameter_validator.py#124\124]8;;\
                             parameters                                                                            

Kedro context loaded from /app
Catalog datasets: ['raw_employees', 'bronze_employees', 'silver_employees', 'gold_employees', 'raw_sales', 'bronze_sales', 'silver_sales', 'gold_sales', 'raw_inventory', 'bronze_inventory', 'silver_inventory', 'gold_inventory', 'parameters']


                    WARNING  /usr/local/lib/python3.12/site-packages/nbformat/validator.py:434:     ]8;id=16264441;file:///usr/local/lib/python3.12/warnings.py\warnings.py]8;;\:]8;id=16264442;file:///usr/local/lib/python3.12/warnings.py#112\112]8;;\
                             MissingIDFieldWarning: Cell is missing an id field, this will become a                
                             hard error in future nbformat versions. You may want to use                           
                             `normalize()` on your notebooks before validations (available since                   
                             nbformat 5.1.4). Previous versions of nbformat are fixing this issue                  
                             transparently, and will stop doing so in the future.                                  
                               _validate(nbdict, ref, version, version_minor, relax_add_props)                     
                                                                                                                   

In [2]:
bronze_employees = catalog.load("bronze_employees")
bronze_employees

                    INFO     Loading data from bronze_employees (CSVDataset)...                ]8;id=16264449;file:///usr/local/lib/python3.12/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=16264450;file:///usr/local/lib/python3.12/site-packages/kedro/io/data_catalog.py#1050\1050]8;;\

,employee_id,full_name,department,salary,hire_date,internal_notes
0,101.0,Alice Johnson,engineering,95000,2021-03-15,top performer
1,102.0,Bob Smith,SALES,72000,2020-07-01,NaN
2,103.0,Carla Diaz,engineering,88000,2022-01-10,remote
3,NaN,Ghost Row,unknown,0,2019-05-05,"bad data, should be dropped"
4,104.0,David Lee,marketing,68000,2023-11-20,NaN


In [3]:
import pandas as pd

silver_employees = bronze_employees.dropna(subset=["employee_id"]).copy()
silver_employees["employee_id"] = silver_employees["employee_id"].astype(int)
silver_employees["full_name"] = clean_text(silver_employees["full_name"])
silver_employees["department"] = clean_text(silver_employees["department"]).str.lower()
silver_employees["hire_date"] = pd.to_datetime(silver_employees["hire_date"]).dt.date
silver_employees

,employee_id,full_name,department,salary,hire_date,internal_notes
0,101,Alice Johnson,engineering,95000,2021-03-15,top performer
1,102,Bob Smith,sales,72000,2020-07-01,NaN
2,103,Carla Diaz,engineering,88000,2022-01-10,remote
4,104,David Lee,marketing,68000,2023-11-20,NaN


In [4]:
catalog.save("silver_employees", silver_employees)

                    INFO     Saving data to silver_employees (CSVDataset)...                   ]8;id=16264456;file:///usr/local/lib/python3.12/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=16264457;file:///usr/local/lib/python3.12/site-packages/kedro/io/data_catalog.py#1006\1006]8;;\

## PySpark alternative (reference only)

PySpark isn't installed in this image. Left commented out to show how this
stage would transform with Spark instead of pandas.

In [5]:
# from pyspark.sql.functions import col, lower, to_date, trim
#
# bronze_employees_spark = (
#     spark.read.option("header", "true")
#     .csv(str(PROJECT_ROOT / "data/02_bronze/employees.csv"))
# )
#
# silver_employees_spark = (
#     bronze_employees_spark
#     .filter(col("employee_id").isNotNull())
#     .withColumn("employee_id", col("employee_id").cast("int"))
#     .withColumn("full_name", trim(col("full_name")))
#     .withColumn("department", lower(trim(col("department"))))
#     .withColumn("hire_date", to_date(col("hire_date")))
# )
#
# silver_employees_spark.write.mode("overwrite").option("header", "true").csv(
#     str(PROJECT_ROOT / "data/03_silver/employees.csv")
# )